In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Pusa_Delhi_IMD_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,214.0,103.0,135.0,63.0,92.0,61.0,NaN,NaN,100.0,108.0,326.0,NaN
1,2,327.0,135.0,170.0,76.0,64.0,82.0,NaN,NaN,101.0,103.0,NaN,NaN
2,3,341.0,130.0,112.0,107.0,76.0,89.0,NaN,NaN,103.0,103.0,428.0,NaN
3,4,282.0,191.0,97.0,71.0,NaN,116.0,NaN,150.0,103.0,110.0,NaN,NaN
4,5,303.0,194.0,101.0,91.0,NaN,120.0,NaN,63.0,NaN,119.0,392.0,NaN
5,6,362.0,156.0,97.0,92.0,180.0,87.0,NaN,129.0,101.0,NaN,374.0,NaN
6,7,334.0,200.0,114.0,94.0,116.0,252.0,NaN,102.0,102.0,124.0,NaN,NaN
7,8,336.0,105.0,173.0,118.0,104.0,NaN,NaN,117.0,81.0,104.0,376.0,NaN
8,9,418.0,144.0,86.0,150.0,142.0,120.0,NaN,92.0,76.0,104.0,371.0,283.0
9,10,385.0,129.0,124.0,104.0,164.0,131.0,NaN,108.0,NaN,136.0,NaN,NaN


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    34 non-null     float64
 2   February   32 non-null     float64
 3   March      36 non-null     float64
 4   April      33 non-null     float64
 5   May        33 non-null     float64
 6   June       21 non-null     float64
 7   July       2 non-null      float64
 8   August     16 non-null     float64
 9   September  20 non-null     float64
 10  October    32 non-null     float64
 11  November   28 non-null     float64
 12  December   24 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,214.0,103.0,135.0,63.0,92.000000,89.0,15.5,89.25,100.0,108.0,326.000000,246.75
1,2,327.0,135.0,170.0,76.0,64.000000,82.0,15.5,89.25,101.0,103.0,274.785714,246.75
2,3,341.0,130.0,112.0,107.0,76.000000,89.0,15.5,89.25,103.0,103.0,428.000000,246.75
3,4,282.0,191.0,97.0,71.0,112.545455,89.0,15.5,89.25,103.0,110.0,274.785714,246.75
4,5,303.0,194.0,101.0,91.0,112.545455,89.0,15.5,89.25,84.1,119.0,392.000000,246.75
